# 02 · Versión A: preguntar al LLM de visión sin entrenarlo (*zero-shot*)

## Qué vas a aprender

- Qué es un **LLM de visión** (VLM) y de qué piezas se compone.
- Qué significa **zero-shot** y qué es un *prompt*.
- Cómo medir una respuesta de un LLM cuando tiene que ser un JSON: **% de JSON válido** y **error por ajuste**.
- A leer respuestas reales, buenas y malas, y a detectar cuándo un modelo «contesta» sin mirar la foto.
- A fijar el punto de partida con el que compararemos el resto: la pregunta del **acto 1** es si un LLM pequeño puede leer el estilo de color de una foto.

**Este notebook** lee las parejas del notebook 01 y deja en la carpeta de trabajo `resultados/pred_A.jsonl` (las respuestas) y `resultados/metricas_A.json` (las métricas).

## 1. Qué es un LLM de visión

Un LLM (modelo de lenguaje grande) es un programa que, dado un texto, escribe la continuación más probable, **trozo a trozo** (cada trozo es una palabra o parte de una palabra, y se llama *token*). Un **LLM de visión** (VLM, *vision-language model*) es uno al que se le ha «pegado un ojo»: además de texto, puede recibir una imagen. Se compone de tres piezas:

```text
foto ──► [1 codificador de imagen] ──► [2 conector] ──► 64 «palabras de imagen» ──┐
                                                                                  ├─► [3 modelo de lenguaje] ──► respuesta
pregunta (texto) ─────────────────────────────────────────────────────────────────┘
```

1. El **codificador de imagen** (aquí SigLIP, 86 M de parámetros) convierte la foto en vectores de números.
2. El **conector** (7 M) comprime esos vectores en 64 «palabras de imagen»: vectores del mismo tamaño que los de las palabras del texto.
3. El **modelo de lenguaje** (aquí SmolLM2, 135 M) lee esas 64 palabras de imagen, seguidas de nuestra pregunta, y escribe la respuesta token a token.

Usamos **SmolVLM-256M-Instruct** (Hugging Face, licencia Apache-2.0): de los VLM más pequeños que hay, cabe en un portátil sin GPU. Ojo con la escala: «pequeño» son 256 millones de parámetros; los asistentes que conoces tienen cientos de veces más.

## 2. Qué es zero-shot y qué es un prompt

- El **prompt** es el texto con el que le pedimos algo al modelo.
- **Zero-shot** («cero disparos») significa que **no le enseñamos ningún ejemplo**: solo le describimos la tarea en el prompt y vemos qué contesta. Es lo más barato que se puede hacer con un LLM: no hay que entrenar nada. Por eso es el primer experimento que conviene hacer, y la medida con la que compararemos todo lo demás: la versión **A**.

El prompt está en **inglés** porque el modelo se entrenó casi todo en inglés; las claves del JSON las dejamos en español porque son las de nuestro programa. Para generar la respuesta usamos `do_sample=False`: el modelo escoge siempre el siguiente token más probable, sin azar, así que **la misma foto da siempre la misma respuesta**.

In [1]:
import os
N_EJEMPLOS = int(os.environ.get("ESTILO_N_EJEMPLOS", "60"))
EPOCAS = int(os.environ.get("ESTILO_EPOCAS", "2"))
RESOLUCION = int(os.environ.get("ESTILO_RESOLUCION", "224"))
MODELO = os.environ.get("ESTILO_MODELO", "HuggingFaceTB/SmolVLM-256M-Instruct")
CARPETA_TRABAJO = os.environ.get("ESTILO_TRABAJO", "_trabajo")

## 3. Cargar el modelo

Cargamos el modelo en CPU (si es la primera vez y no hiciste el notebook 00, también lo descarga: unos 494 MB). Después imprimimos cuántos parámetros tiene cada una de las tres piezas.

**Qué debes ver:** tres líneas con unos 86 M (codificador de imagen), 7 M (conector) y 135 M (modelo de lenguaje) de parámetros, y la versión de `torch` con sufijo `+cpu`.

In [2]:
import json, sys
from collections import Counter
from pathlib import Path
import numpy as np

# Localizar la carpeta del extra (la que contiene `estilo/`), desde donde se ejecute el notebook
raiz = Path(os.environ.get('ESTILO_EXTRA') or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'estilo').is_dir()))
sys.path.insert(0, str(raiz))
import estilo as es

trabajo = es.carpeta_trabajo(CARPETA_TRABAJO)   # fija también HF_HOME si no venía dado
hilos = es.limitar_hilos()
if not (trabajo / 'parejas' / 'parejas.jsonl').exists():
    raise FileNotFoundError('Faltan las parejas: ejecuta antes el notebook 01.')
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| hilos:', hilos)

with es.Cronometro() as c:
    proc, modelo_vlm = es.cargar_vlm(MODELO)
print(f'Modelo {MODELO} cargado en {c.s:.1f} s (incluye la descarga si no estaba en caché)')
for nombre, parte in modelo_vlm.model.named_children():
    print(f'  {nombre:14s} {sum(p.numel() for p in parte.parameters()) / 1e6:6.1f} M parámetros')

torch 2.15.0+cpu | transformers 5.23.0 | hilos: 14


Modelo HuggingFaceTB/SmolVLM-256M-Instruct cargado en 6.0 s (incluye la descarga si no estaba en caché)
  vision_model     86.4 M parámetros
  connector         7.1 M parámetros
  text_model      134.6 M parámetros


## 4. El prompt

Le explicamos qué ajustes queremos, en qué rango, y que conteste **solo** con un JSON. Lo que imprimimos es el prompt tal como lo recibe el modelo, con su **plantilla de chat**: unas marcas especiales (`<|im_start|>User:`, `Assistant:`) que indican quién habla. `<image>` marca el sitio donde entran las 64 palabras de imagen.

**Qué debes ver:** la pregunta completa, con los cinco ajustes y sus rangos, terminando en `Assistant:`. A partir de ahí el modelo escribe.

In [3]:
print(es.vlm.texto_prompt(proc))

<|im_start|>User:<image>Look at the colors of this photo. Estimate the color edits that were applied to it and answer ONLY with a JSON object with these keys: "exposicion" (exposure in stops, -0.7 to 0.7), "temperatura" (white balance, -1 cool/blue to 1 warm/orange), "tinte" (tint, -1 green to 1 magenta), "contraste" (contrast factor, 0.6 to 1.5, 1 = unchanged), "saturacion" (saturation factor, 0.2 to 1.8, 1 = unchanged).<end_of_utterance>
Assistant:


## 5. Primer intento: solo el prompt

Probamos con tres fotos editadas de *test* (las que el modelo nunca ha visto) y limitamos la respuesta a 40 tokens para que sea rápido.

**Qué debes ver:** este modelo tan pequeño **ignora la orden y describe la foto**, algo como «The image shows a dog running on a grassy field...». No escribe ningún JSON. Esto es un fallo de *seguir instrucciones*, no de ver colores.

In [4]:
test = es.leer_parejas(trabajo, 'test')
print('Sin empezar la respuesta (tres fotos):')
for f in test[:3]:
    print('  ->', es.preguntar(proc, modelo_vlm, es.abrir(trabajo, f['editada']), max_tokens=40).replace('\n', ' '))

Sin empezar la respuesta (tres fotos):


  -> The image shows a dog running on a grassy field. The dog appears to be a mix of brown, white, and black, with a white and black coat. The dog's fur is short and


  -> The image shows a dog lying on a blanket, which appears to be a blanket with a pattern of white and gray. The dog is brown and appears to be relaxed, with its tongue hanging out and


  -> The dog's face is slightly turned away from the camera, and its ears are perked up. The dog's fur is mostly white with some black patches, and its eyes are dark brown. The


## 6. Segundo intento: empezar nosotros la respuesta

Un truco legítimo, que **no le enseña nada** al modelo, es **escribir nosotros el principio de la respuesta**: `{"exposicion":`. El modelo ya no puede describir la foto: tiene que seguir escribiendo desde ahí, y lo más natural es continuar el JSON. Esa es la versión A oficial.

Para cada foto editada de *test* generamos la respuesta e intentamos **leer el JSON** con `es.parsear`. Esa función es tolerante (acepta números entre comillas y texto alrededor), recorta cada valor al rango del ajuste y, si falta algún número, pone el valor neutro y marca la respuesta como **formato incorrecto**.

Medimos también el tiempo de cada foto. **Esta celda es la lenta del notebook**: en el estudio previo (portátil de 12 núcleos, sin GPU) tardó unos 5 s por foto, es decir, unos 5 minutos para las 60 de *test*. Mientras tanto no se ve nada: es normal.

In [5]:
INICIO = ' {"exposicion":'
resultados, tiempos = [], []
for f in test:
    with es.Cronometro() as c:
        respuesta = es.preguntar(proc, modelo_vlm, es.abrir(trabajo, f['editada']), inicio=INICIO)
    tiempos.append(c.s)
    pred, completo = es.parsear(respuesta)
    resultados.append({'id': f['id'], 'respuesta': respuesta, 'pred': pred, 'completo': completo})

(trabajo / 'resultados').mkdir(exist_ok=True)
(trabajo / 'resultados' / 'pred_A.jsonl').write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False) for r in resultados), encoding='utf-8')
print(f'Versión A: {len(test)} fotos; {np.mean(tiempos):.2f} s por foto de media (mín {min(tiempos):.2f}, máx {max(tiempos):.2f})')

Versión A: 60 fotos; 5.61 s por foto de media (mín 3.52, máx 9.97)


## 7. ¿Cuánto se equivoca? Las métricas

- **% de JSON válido** (formato correcto): proporción de respuestas en las que encontramos los cinco números. Mide si el modelo *sabe escribir* lo que le pedimos.
- **MAE** (error medio absoluto) de cada ajuste: la media de |valor predicho − valor real| en las fotos de *test*. Mide si el modelo *acierta*.
- **MAE normalizado**: el MAE dividido por el ancho del rango del ajuste (la exposición va de −0,7 a 0,7; el contraste, de 0,6 a 1,5). Así los cinco ajustes se pueden comparar y promediar. **0 es perfecto.**

Un número suelto no dice nada: hace falta una referencia. Usamos dos, que no miran la foto:

- **neutro**: contestar siempre «no se tocó nada» (exposición 0, temperatura 0, contraste 1...).
- **media de train**: contestar siempre el valor medio de cada ajuste en las fotos de entrenamiento.

Si el modelo no mejora a un «modelo» que contesta lo mismo para cualquier foto, **no está leyendo la foto**.

**Cifras de referencia del estudio previo** (20/11/2026, 60 fotos de *test*): formato correcto **55 %**; MAE normalizado medio **0,433**; neutro y media de train, **0,259** los dos; 5,05 s por foto; memoria máxima del proceso 1541 MB. Tus números deberían ser los mismos (la respuesta no tiene azar) salvo los tiempos y la memoria, que dependen de tu equipo.

In [6]:
reales = [f['ajustes'] for f in test]
m = es.error_por_ajuste([r['pred'] for r in resultados], reales)
neutro = es.error_por_ajuste([es.NEUTRO] * len(test), reales)
media = es.error_por_ajuste([es.ajustes_medios(es.leer_parejas(trabajo, 'train'))] * len(test), reales)
m['formato_correcto'] = es.porcentaje_json_valido([r['respuesta'] for r in resultados])
m['segundos_por_foto'] = float(np.mean(tiempos))
m['memoria_pico_mb'] = es.memoria_mb()
m['hilos'] = hilos

print(f"JSON válido (formato correcto): {m['formato_correcto']:.0%}")
print(f"{'ajuste':12s} {'MAE':>6s} {'MAE norm.':>10s} {'neutro':>8s} {'media train':>12s}")
for k in es.NOMBRES:
    print(f"{k:12s} {m['mae_' + k]:6.3f} {m['maen_' + k]:10.3f} {neutro['maen_' + k]:8.3f} {media['maen_' + k]:12.3f}")
print(f"{'MEDIA':12s} {'':6s} {m['mae_normalizado_medio']:10.3f} {neutro['mae_normalizado_medio']:8.3f} {media['mae_normalizado_medio']:12.3f}")
print('Memoria máxima del proceso (MB):', m['memoria_pico_mb'])

# Se guarda en JSON; el run de MLflow de A lo crea el notebook 04 (así, en CI, hay una sola base de MLflow)
(trabajo / 'resultados' / 'metricas_A.json').write_text(json.dumps(m, indent=1), encoding='utf-8')

JSON válido (formato correcto): 55%
ajuste          MAE  MAE norm.   neutro  media train
exposicion    0.683      0.488    0.262        0.262
temperatura   0.984      0.492    0.250        0.251
tinte         0.865      0.432    0.249        0.253
contraste     0.335      0.372    0.242        0.240
saturacion    0.608      0.380    0.290        0.292
MEDIA                    0.433    0.259        0.259
Memoria máxima del proceso (MB): 1518.6


548

**Cómo leer la tabla.** En la columna `MAE norm.` compara cada ajuste con las dos referencias de la derecha. Si el modelo tiene un error **mayor** que el de «neutro» o el de «media de train», contestar siempre lo mismo habría sido mejor que mirar la foto. En el estudio previo ocurrió en los cinco ajustes (0,433 frente a 0,259 de media).

## 8. Respuestas reales: buenas y malas

Mostramos tal cual lo que escribió el modelo (con nuestro arranque `{"exposicion":` incluido), junto al JSON real. «Buenas» son las que tienen el formato correcto, aunque los números estén mal; «malas», las que no se pudieron leer enteras.

**Qué debes ver, y en qué fijarte:** que el JSON se parece a lo pedido, pero **los números no dependen de la foto**: repiten los extremos de los rangos que le dimos en el prompt (0.7, -1, 0.2...), a veces con texto como «-1 green to 1 magenta» o con claves inventadas. En el estudio previo, tres fotos muy distintas recibían prácticamente la misma respuesta. Eso es *copiar el enunciado*, no mirar la foto.

In [7]:
buenas = [r for r in resultados if r['completo']][:3]
malas = [r for r in resultados if not r['completo']][:3]
por_id = {f['id']: f for f in test}
for titulo, grupo in (('BUENAS (formato correcto)', buenas), ('MALAS (no se pudo leer el JSON)', malas)):
    print('=' * 20, titulo, '=' * 20)
    for r in grupo:
        print('real     :', es.a_json(por_id[r['id']]['ajustes']))
        print('respuesta:', r['respuesta'][:300].replace('\n', ' '))
        print()
    if not grupo:
        print('(ninguna)\n')

==================== BUENAS (formato correcto) ====================
real     : {"exposicion": 0.45, "temperatura": 0.5, "tinte": 0.03, "contraste": 1.35, "saturacion": 1.69}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "contraste": "-1", "saturacion": "0.2", "tinte_saturation": "1.8", "tinte_contraste": "1.5", "saturacion_contraste": "1.5", "satur

real     : {"exposicion": 0.0, "temperatura": -0.39, "tinte": 0.35, "contraste": 1.23, "saturacion": 0.43}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "tinte_contraste": "-1", "saturacion": "0.2", "tinte_contraste_saturation": "1", "saturacion_saturation_color": "0.2", "tinte_color_saturation

real     : {"exposicion": 0.37, "temperatura": 0.84, "tinte": 0.99, "contraste": 0.67, "saturacion": 1.58}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "tinte_contraste": "0.6", "saturacion": "0.2", "tinte_contraste_saturation": "1", "saturation_factor": "0.2", "tinte_contraste_sa

Para no quedarnos con tres ejemplos, contamos en las 60 respuestas cuántas son distintas entre sí y cuántas repiten los extremos del prompt (exposición 0,7 y temperatura −1). Si el modelo mirara cada foto, esperaríamos casi tantas respuestas distintas como fotos.

In [8]:
claves = [tuple(r['pred'].values()) for r in resultados]
repetida, veces = Counter(claves).most_common(1)[0]
copian = sum(r['pred']['exposicion'] == 0.7 and r['pred']['temperatura'] == -1.0 for r in resultados)
print(f'{len(set(claves))} respuestas distintas en {len(resultados)} fotos')
print(f'La más repetida sale {veces} veces: {es.a_json(dict(zip(es.NOMBRES, repetida)))}')
print(f'{copian} respuestas con exposición 0,7 y temperatura -1 (los extremos del prompt)')

5 respuestas distintas en 60 fotos
La más repetida sale 28 veces: {"exposicion": 0.7, "temperatura": -1.0, "tinte": -1.0, "contraste": 0.6, "saturacion": 0.2}
60 respuestas con exposición 0,7 y temperatura -1 (los extremos del prompt)


## Qué te llevas

- Un **LLM de visión** es un LLM con un ojo: codificador de imagen + conector + modelo de lenguaje.
- **Zero-shot** es preguntar sin enseñar nada. Es barato y siempre se debe probar primero, aunque solo sea para tener un punto de partida.
- Un LLM tiene que dominar **dos cosas distintas**: escribir el *formato* pedido y *acertar el contenido*. Se miden por separado (% de JSON válido y error por ajuste).
- Un error solo significa algo **frente a una referencia**. Aquí, las dos referencias («neutro» y «media de train») ni siquiera miran la foto.

## Adelanto del acto 1: ¿lo hará mejor con fine-tuning?

Lo natural ahora es pensar: «el modelo no sabe hacer esto porque nadie se lo ha enseñado. Si lo **afinamos** con las parejas del notebook 01, aprenderá». En el notebook 03 lo comprobamos con **LoRA**: entrenar unos pocos millones de parámetros encima del modelo congelado. Hay dos preguntas distintas, y conviene no mezclarlas:

1. ¿Aprenderá el **formato**? (Es de esperar que sí: escribir un JSON es fácil de aprender.)
2. ¿Aprenderá la **tarea**, es decir, a bajar del error de «media de train» (0,259 en el estudio previo)? Esa es la que importa.

Apunta tu apuesta antes de pasar al 03.

## Para pensar

1. El MAE normalizado de A es mayor que el de «neutro». ¿Qué te dice eso sobre si el modelo «sabe» de colores? ¿Y sobre si merece la pena usarlo tal cual?
2. Hemos empezado nosotros la respuesta con `{"exposicion":`. ¿Es trampa? ¿Cambia lo que el modelo sabe, o solo lo que escribe? ¿Qué pasaría con el % de JSON válido sin ese arranque?